# Tinjau dataset utama
Pilih kernel virtualenv proyek lalu jalankan sel dari atas ke bawah. Notebook ini hanya membaca CSV lokal; tidak memanggil SerpApi/Gemini. Saat pengumpulan berlangsung, jalankan ulang sel pemuatan untuk melihat ekspor terakhir (diperbarui setelah tiap query selesai).
Aturan: tiga percobaan per query, minimal dua valid. Data masih berupa kandidat artikel, belum siap pemodelan. Panduan: `docs/MAIN_DATASET.md`.

In [1]:
from pathlib import Path
from collections import Counter
from html import escape
import csv
from IPython.display import HTML, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/collect_main_dataset.py').exists())
DATASET_ID = 'main_01'
OUTPUT = ROOT / 'data/interim/main' / DATASET_ID

def read_table(name):
    path = OUTPUT / (name + '.csv')
    if not path.exists():
        print(f'{name}: belum diekspor; tunggu query pertama selesai.')
        return []
    with path.open(encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))

def show(rows, columns, limit=50):
    head = ''.join('<th>' + escape(c) + '</th>' for c in columns)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(r.get(c, ''))) + '</td>' for c in columns) + '</tr>' for r in rows[:limit])
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))
    print(f'Ditampilkan {min(limit, len(rows))} dari {len(rows)} baris.')

In [2]:
# Jalankan ulang sel ini untuk memperbarui data yang dibaca.
queries = read_table('queries')
trials = read_table('trials')
pairs_google = read_table('query_article_pairs')
pairs = read_table('query_article_pairs_union')
sources = read_table('sources')
print('Query terjadwal:', len(queries))
print('Query selesai:', sum(r['collection_complete'] == 'True' for r in queries))
print('Query selesai dengan minimal 2 valid:', sum(r['collection_complete'] == 'True' and r['grounding_eligible'] == 'True' for r in queries))
print('Status percobaan:', dict(Counter(r['status'] for r in trials)))
print('Pasangan dari Google Top-10:', len(pairs_google))
print('Pasangan gabungan Google + semua sitasi valid Gemini:', len(pairs))
print('Asal kandidat (untuk audit):', dict(Counter(r['candidate_origin'] for r in pairs)))
print('Status pencocokan/label:', dict(Counter(r['label_status'] for r in pairs)))
union_counts = Counter(r['query_id'] for r in pairs)
for row in queries:
    row['n_candidate_union'] = union_counts[row['query_id']]
show(queries, ['query_text', 'domain', 'google_status', 'n_google_results', 'n_candidate_union', 'n_trial_records', 'n_valid', 'grounding_eligible', 'collection_complete', 'within_time_window'])

Query terjadwal: 41
Query selesai: 23
Query selesai dengan minimal 2 valid: 21
Status percobaan: {'completed': 70, 'error': 1}
Pasangan dari Google Top-10: 202
Pasangan gabungan Google + semua sitasi valid Gemini: 554
Asal kandidat (untuk audit): {'google_only': 132, 'google_and_gemini': 70, 'gemini_only': 352}
Status pencocokan/label: {'threshold_not_set': 404, 'url_matching_uncertain': 109, 'query_not_eligible_or_incomplete': 41}


query_text,domain,google_status,n_google_results,n_candidate_union,n_trial_records,n_valid,grounding_eligible,collection_complete,within_time_window
Minum apa agar batuk cepat sembuh?,kesehatan,completed,9,21,3,3,True,True,True
Gaji 6 juta pajak berapa?,keuangan,completed,8,8,3,0,False,True,True
VPN itu buat apa sih?,teknologi,completed,7,12,3,3,True,True,True
Apa saja merk obat batuk yang aman untuk ibu hamil?,kesehatan,completed,8,30,3,3,True,True,True
Apakah gaji 4 juta wajib pajak?,keuangan,completed,9,23,3,3,True,True,True
Cara mengaktifkan VPN gimana?,teknologi,completed,7,29,3,3,True,True,True
Obat batuk yang bagus merk apa?,kesehatan,completed,9,36,3,3,True,True,True
NPWP itu apa sih?,keuangan,completed,8,31,3,3,True,True,True
Apa nama VPN gratis?,teknologi,completed,9,35,3,3,True,True,True
Apa obat batuk yang aman untuk penderita diabetes dan jantung?,kesehatan,completed,9,34,3,2,True,True,True


Ditampilkan 41 dari 41 baris.


## Periksa satu query
Isi `QUERY_TEXT` dengan teks persis dari tabel, atau biarkan kosong untuk memakai query pertama yang sudah menghasilkan pasangan. Satu baris mewakili satu pasangan query-artikel; satu query bisa memiliki banyak baris. Kandidat merupakan gabungan Google Top-10 dan semua URL tujuan yang disitasi pada percobaan Gemini valid. URL berulang per query digabung. `citation_proportion` kosong berbeda dari nol: periksa `n_valid`, `n_unknown_matches`, dan `label_status`. `source_url` memuat URL tujuan yang berhasil diketahui; `raw_url` menyimpan redirect asli untuk audit. `candidate_origin` adalah metadata audit, bukan fitur prediksi.

In [ ]:
QUERY_TEXT = ''
selected_query = QUERY_TEXT or (pairs[0]['query_text'] if pairs else '')
selected_ids = {r['query_id'] for r in queries if r['query_text'] == selected_query}
print(selected_query or 'Belum ada pasangan kandidat.')
show([r for r in trials if r['query_id'] in selected_ids], ['repetition', 'status', 'valid_grounding', 'finish_reason', 'n_cited_sources', 'error'])
show([r for r in pairs if r['query_id'] in selected_ids], ['candidate_origin', 'google_positions', 'article_title', 'article_url', 'n_valid', 'n_cited', 'n_unknown_matches', 'citation_proportion', 'citation_lower', 'citation_upper', 'label_status'], limit=len(pairs))
show([r for r in sources if r['query_id'] in selected_ids], ['trial_id', 'is_cited', 'title', 'source_url', 'resolution_status'], limit=len(sources))

In [ ]:
# Daftar tindak lanjut; query yang belum selesai ditinjau setelah pengumpulan.
needs_review = [r for r in queries if r['collection_complete'] == 'True' and (r['grounding_eligible'] != 'True' or r['within_time_window'] != 'True')]
show(needs_review, ['query_text', 'domain', 'n_valid', 'within_time_window'])
missing_urls = [r for r in sources if r['is_cited'] == 'True' and not r['source_url']]
show(missing_urls, ['query_id', 'trial_id', 'title', 'resolution_status'])
print('Artikel masih memerlukan crawling, seleksi halaman, ekstraksi fitur, dan keputusan ambang label.')